# Edge Measurement & Walk-Forward Validation
Tags signals with forward returns and measures hit-rate, avg R, and payoff ratio.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings; warnings.filterwarnings('ignore')

from data_fetch.data_fetch import fetch_ohlcv
from algos.pivot_engine import extrems, tag_signals
from algos.confluence   import measure_edge
print('✅ Ready')

In [ ]:
TICKER   = 'RELIANCE.NS'
INTERVAL = '1d'
DAYS     = 1000
ORDER    = 5
HORIZONS = (5, 10, 20)

In [ ]:
df     = fetch_ohlcv(TICKER, interval=INTERVAL, days=DAYS)
pivots = extrems(df, order=ORDER)
tagged = tag_signals(df, pivots, horizons=HORIZONS)
print(f'{len(tagged)} pivot signals tagged')
tagged.tail()

In [ ]:
edge = measure_edge(tagged, horizons=HORIZONS)
print('\n── Edge Metrics ──')
display(edge)

In [ ]:
# Plot forward return distribution
for h in HORIZONS:
    col = f'fwd_{h}'
    if col in tagged.columns:
        fig = px.histogram(
            tagged.dropna(subset=[col]),
            x=col, color='type',
            nbins=40, barmode='overlay',
            title=f'Forward Return Distribution – {h}-bar horizon',
            template='plotly_dark',
            color_discrete_map={'B':'#22c55e','T':'#ef4444'},
        )
        fig.update_layout(height=350)
        fig.show()

In [ ]:
# Walk-forward split: first 70% train, last 30% test
split = int(len(pivots) * 0.7)
train_tagged = tag_signals(df, pivots[:split], horizons=HORIZONS)
test_tagged  = tag_signals(df, pivots[split:], horizons=HORIZONS)

train_edge = measure_edge(train_tagged, horizons=HORIZONS)
test_edge  = measure_edge(test_tagged,  horizons=HORIZONS)

print('── Train edge ──'); display(train_edge)
print('── Test edge  ──'); display(test_edge)